In [2]:
import os
import json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

folder = "../res_expe_12h"  # Dossier contenant les fichiers JSON

# --- 1. Extraction des données avec les bonnes clés ---
data = []

for filename in os.listdir(folder):
    if filename.endswith(".json"):
        filepath = os.path.join(folder, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            file_content = json.load(f)
            
            # Gère le cas où le JSON contient une liste ou un seul dict
            experiences = file_content if isinstance(file_content, list) else [file_content]
                
            for exp in experiences:
                try:
                    # 1. Extraction du ratio
                    params = exp.get("parameters", {})
                    ratio = params.get("ratioRandom", 0)
                    
                    # 2. Extraction du score Solveur
                    results = exp.get("results", {})
                    solver = results.get("solver", {})
                    solver_score = solver.get("score", 0)
                    
                    # 3. Extraction du score MCTS (Moyenne du dernier step de chaque essai)
                    mcts = results.get("mcts", {})
                    mcts_tries = mcts.get("tries", [])
                    
                    mcts_final_scores = []
                    for t in mcts_tries:
                        steps = t.get("steps", [])
                        if steps:
                            # On prend le score de la toute dernière étape (step)
                            mcts_final_scores.append(steps[-1].get("score", 0))
                            
                    mcts_score_moyen = sum(mcts_final_scores) / len(mcts_final_scores) if mcts_final_scores else 0
                    
                    # Ajout des données extraites à notre liste
                    data.append({
                        "Fichier": filename,
                        "Ratio": ratio,
                        "Solveur_Score": solver_score,
                        "MCTS_Score_Moyen": mcts_score_moyen
                    })
                except Exception as e:
                    print(f"⚠️ Erreur lors de l'extraction de {filename} : {e}")

# --- 2. Création et préparation du DataFrame ---
df = pd.DataFrame(data)

if not df.empty:
    # Sécurité pour éviter la division par zéro
    df = df[df['Solveur_Score'] > 0].copy()
    
    # Calcul de l'erreur normalisée (%)
    df['Erreur_%'] = ((df['Solveur_Score'] - df['MCTS_Score_Moyen']) / df['Solveur_Score']) * 100

    # On regroupe par Ratio pour moyenner les résultats s'il y a plusieurs fichiers pour un même ratio
    df_grouped = df.groupby('Ratio').mean(numeric_only=True).reset_index().sort_values('Ratio')

    # --- 3. Création des graphiques ---
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Impact du Ratio Random sur le Score", "Erreur Normalisée Moyenne du MCTS (%)")
    )

    # Graphique 1 : Score Solveur vs MCTS
    fig.add_trace(go.Scatter(
        x=df_grouped['Ratio'], y=df_grouped['Solveur_Score'], 
        mode='lines+markers', name='Solveur (Optimum)',
        line=dict(color='red', dash='dot', width=3),
        marker=dict(size=10)
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=df_grouped['Ratio'], y=df_grouped['MCTS_Score_Moyen'], 
        mode='lines+markers', name='MCTS (Moyenne)',
        line=dict(color='blue', width=3),
        marker=dict(size=10)
    ), row=1, col=1)

    # Graphique 2 : Erreur Normalisée (Histogramme par Ratio)
    fig.add_trace(go.Bar(
        x=df_grouped['Ratio'].astype(str), 
        y=df_grouped['Erreur_%'], 
        name='Erreur Moyenne (%)',
        marker_color='orange'
    ), row=1, col=2)

    fig.add_hline(y=0, line_dash="dash", line_color="green", annotation_text="Optimum (Erreur = 0%)", row=1, col=2)

    # --- 4. Mise en page globale ---
    fig.update_layout(
        title_text="Analyse globale des résultats MCTS vs Solveur",
        template="plotly_white",
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.1, xanchor="right", x=1)
    )

    # Configuration des axes
    fig.update_xaxes(title_text="Ratio Random", row=1, col=1)
    fig.update_yaxes(title_text="Score Absolu", row=1, col=1)
    fig.update_xaxes(title_text="Ratio Random", type='category', row=1, col=2) # Category pour bien centrer les barres
    fig.update_yaxes(title_text="Erreur par rapport au Solveur (%)", row=1, col=2)

    # Affichage
    fig.show()
else:
    print("❌ Aucune donnée n'a pu être extraite des fichiers.")

In [5]:
import os
import json
import pandas as pd
import plotly.express as px

folder = "../res_expe_12h"  # Dossier contenant les fichiers JSON

# --- 1. Extraction des données ---
data_convergence = []
solver_scores = {}  # Pour mémoriser le score du solveur de chaque configuration

for filename in os.listdir(folder):
    if filename.endswith(".json"):
        filepath = os.path.join(folder, filename)
        with open(filepath, 'r', encoding='utf-8') as f:
            file_content = json.load(f)
            
            # Gère liste ou dictionnaire unique
            experiences = file_content if isinstance(file_content, list) else [file_content]
                
            for exp in experiences:
                try:
                    ratio = exp.get("parameters", {}).get("ratioRandom", 0)
                    
                    # Identifiant unique pour la courbe (utile si plusieurs fichiers ont le même ratio)
                    label = f"Ratio {ratio} ({filename})"
                    
                    # Récupération du score du solveur
                    solver_score = exp.get("results", {}).get("solver", {}).get("score", None)
                    if solver_score is not None:
                        solver_scores[label] = solver_score
                        
                    # Récupération de l'évolution du score MCTS (on prend le 1er essai)
                    mcts_tries = exp.get("results", {}).get("mcts", {}).get("tries", [])
                    if mcts_tries:
                        steps = mcts_tries[0].get("steps", [])
                        for step_idx, step in enumerate(steps):
                            data_convergence.append({
                                "Configuration": label,
                                "Etape": step_idx,
                                "Score": step.get("score", 0)
                            })
                            
                except Exception as e:
                    print(f"⚠️ Erreur sur {filename} : {e}")

# --- 2. Création du DataFrame ---
df_convergence = pd.DataFrame(data_convergence)

if not df_convergence.empty:
    # --- 3. Création du graphique de convergence ---
    fig = px.line(
        df_convergence, 
        x="Etape", 
        y="Score", 
        color="Configuration",
        title="Courbe de convergence du MCTS vs Optimum du Solveur",
        labels={"Etape": "Nombre d'itérations (Steps)", "Score": "Score", "Configuration": "Configuration"}
    )
    
    # --- 4. Ajout des droites horizontales (Solveur) ---
    for trace in fig.data:
        label = trace.name
        if label in solver_scores:
            fig.add_hline(
                y=solver_scores[label], 
                line_dash="dash", 
                line_color=trace.line.color,  # Utilise la même couleur que la courbe MCTS
                annotation_text=f"Optimum Solveur",
                annotation_position="bottom right"
            )

    # --- 5. Mise en page et déplacement de la légende à droite ---
    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        # Légende positionnée verticalement sur la droite de la figure
        legend=dict(
            title="Légende",
            x=1.02,          # Légèrement à droite de la zone graphique
            y=1,             # Aligné en haut
            xanchor="left",
            yanchor="top"
        ),
        # Augmentation de la marge de droite pour laisser de la place à la légende
        margin=dict(t=60, b=40, l=40, r=200)
    )
    
    fig.show()
else:
    print("❌ Aucune donnée de convergence n'a pu être extraite.")